![Banner](../../images/00_banner.png)


# 0.2 Spatial Data Types and Containers


**Part:** 0:  Setup and Foundations (Chapter 0.2 of 4)

**Dataset:** US counties / Northeast diabetes lattice (`data/US_COUNTY.shp`, `data/diabetes_northeast.shp`)

**Focus:** Three spatial data types, support and MAUP, and the `SpatialData` container with roles

**Library:** `spatialstats.core` (`SpatialData`)

**Companion:** [`ch00_setup_and_foundations.ipynb`](ch00_setup_and_foundations.ipynb)


***

## Table of Contents

1. [Overview](#1.-Overview)

2. [Python Setup](#2.-Python-Setup)

3. [What is Spatial Data?](#3.-What-is-Spatial-Data?)

4. [Every Spatial Dataset Has Three Parts](#4.-Every-Spatial-Dataset-Has-Three-Parts)

5. [Spatial Data Containers](#5.-Spatial-Data-Containers)

6. [`SpatialData`: a Table with Roles](#6.-SpatialData:-a-Table-with-Roles)

7. [Reprojection Helpers](#7.-Reprojection-Helpers)

8. [Summary and Where Next](#8.-Summary-and-Where-Next)


***
## 1. Overview

Before loading a file into an analysis, you need to know **what kind of spatial data it is** and how
PySpatialStats expects roles (coordinates, response, covariates) to be declared.

| Step | What we do |
|------|------------|
| Types | Distinguish geostatistical fields, lattice (areal) data, and point patterns |
| Anatomy | Attributes, geometry, and the CRS as the three parts of every layer |
| `SpatialData` | Assign roles; see what `coords_array()` really returns |
| CRS helpers | `to_crs` / `ensure_projected` on a `SpatialData` object |

**Learning objectives (this notebook).** Distinguish the three basic types of spatial data and the
model behind each; load and organise data with GeoPandas and `SpatialData`; know what `coords_array()` returns.


***
## 2. Python Setup


In [ ]:
import ast, importlib, importlib.util, inspect, platform, sys, time, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)


def find_repo_root(start: Path = Path.cwd()) -> Path:
    """Walk up from ``start`` until the folder containing ``pyproject.toml`` and ``data/`` is found."""
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists() and (p / "data").is_dir():
            return p
    raise FileNotFoundError("Run this notebook from inside the PySpatialStats repository.")


ROOT = find_repo_root()
DATA = ROOT / "data"
try:
    import spatialstats
except ImportError:                       # package not pip-installed: fall back to the source tree
    sys.path.insert(0, str(ROOT))
    import spatialstats

plt.rcParams.update({
    "figure.dpi": 100, "savefig.dpi": 100, "font.size": 10, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "legend.frameon": False,
})
C = {"blue": "#2b6cb0", "red": "#c0392b", "green": "#2f855a", "orange": "#dd6b20", "grey": "#718096", "purple": "#6b46c1"}

print(f"working directory : {Path.cwd().name}/")
print(f"repository data   : {DATA.relative_to(ROOT)}/  ({len(list(DATA.iterdir()))} files)")
print(f"spatialstats      : version {spatialstats.__version__}")


## 3. What is Spatial Data?

### Three data types, three statistical models

Spatial statistics is organised by *what is random* and *what is fixed*. Let $D\subset\mathbb{R}^2$ be the study region and
$\mathbf{s}=(x,y)\in D$ a location.

**(a) Geostatistical (point-referenced) data.** A quantity exists everywhere and is sampled at chosen locations.
The model is a random field
$$
\{Z(\mathbf{s}) : \mathbf{s}\in D\},\qquad D \text{ continuous},
$$
observed at $\mathbf{s}_1,\dots,\mathbf{s}_n$. Its first two moments are the *mean function* and *covariance function*
$$
\mu(\mathbf{s}) = \mathbb{E}\,Z(\mathbf{s}),\qquad C(\mathbf{s},\mathbf{s}')=\operatorname{Cov}\big(Z(\mathbf{s}),Z(\mathbf{s}')\big).
$$
Under **second-order stationarity** $\mu$ is constant and $C(\mathbf{s},\mathbf{s}+\mathbf{h})=C(\mathbf{h})$ depends only on the
separation $\mathbf{h}$; the *semivariogram* $\gamma(\mathbf{h})=\tfrac12\operatorname{Var}\big[Z(\mathbf{s}+\mathbf{h})-Z(\mathbf{s})\big]=C(\mathbf{0})-C(\mathbf{h})$
is then the workhorse of interpolation (**Chapter 7**). *Example: PM2.5 at monitoring stations.*

**(b) Lattice (areal) data.** The locations are a fixed, discrete set of regions $A_1,\dots,A_n$ that tile $D$
(counties, tracts). The data are a vector $\mathbf{z}=(Z_1,\dots,Z_n)^\top$, and what matters is *which regions are neighbours*,
encoded in an $n\times n$ **spatial weights matrix** $\mathbf{W}=(w_{ij})$ (**Chapters 2–6**). *Example: diabetes prevalence by county.*

**(c) Point patterns.** The *locations themselves* are random. A pattern is a random set of points $\mathbf{X}\subset D$, described by
its counting measure $N(B)=\#\{\mathbf{X}\cap B\}$ and its **intensity**
$$
\lambda(\mathbf{s})=\lim_{|d\mathbf{s}|\to 0}\frac{\mathbb{E}\,N(d\mathbf{s})}{|d\mathbf{s}|}.
$$
The reference model is *complete spatial randomness* (CSR): a homogeneous Poisson process, $N(B)\sim\text{Poisson}(\lambda|B|)$ (**Chapter 1**).
*Example: crime incidents.*

The three cover the same map differently: the same crime data could be a point pattern (locations), a lattice (counts per
neighbourhood) or a field (smoothed risk surface). Choosing the representation is the first modelling decision.


In [ ]:
uc = gpd.read_file(DATA / "US_COUNTY.shp")
ca_counties = uc[uc["STATE"] == "California"]

pm = pd.read_csv(DATA / "CA_pm25_2025.csv")
sites = (pm.groupby("Site ID").agg(lat=("Lat", "mean"), lon=("Long", "mean"), pm25=("Daily_PM2.5_ug_m3", "mean"))
           .reset_index())
sites = gpd.GeoDataFrame(sites, geometry=gpd.points_from_xy(sites["lon"], sites["lat"]), crs=4326).to_crs(uc.crs)

city = gpd.read_file(DATA / "buffalo_city.shp")
crimes = gpd.read_file(DATA / "Crime_location_2018.shp")
crimes_sub = crimes.iloc[np.random.default_rng(0).choice(len(crimes), 3000, replace=False)]
# For drawing we use UTM 17N, the correct zone for Buffalo (section 0.5 explains why the files' own CRS is not used)
city_map = city.to_crs(4326).to_crs(26917)
crimes_map = crimes_sub.to_crs(4326).to_crs(26917)

fig, axes = plt.subplots(1, 3, figsize=(14, 4.8))
ca_counties.boundary.plot(ax=axes[0], color="#cbd5e0", lw=0.5)
sites.plot(ax=axes[0], column="pm25", cmap="YlOrRd", markersize=22, edgecolor="k", linewidth=0.3, legend=True,
           legend_kwds={"shrink": 0.6, "label": "annual mean PM2.5 (µg/m³)"})
axes[0].set_title(f"(a) Geostatistical: Z(s) at {len(sites)} stations")

ne.plot(ax=axes[1], column="Dia_pct", cmap="viridis", edgecolor="white", linewidth=0.2, legend=True,
        legend_kwds={"shrink": 0.6, "label": "diabetes prevalence (%)"})
axes[1].set_title(f"(b) Lattice: Z_i for {len(ne)} counties")

city_map.boundary.plot(ax=axes[2], color="k", lw=0.6)
crimes_map.plot(ax=axes[2], markersize=1.5, color=C["red"], alpha=0.5)
axes[2].set_title("(c) Point pattern: 3,000 of 14,001 crimes")
for a in axes:
    a.set_axis_off()
plt.tight_layout(); plt.show()

## 4. Every Spatial Dataset Has Three Parts

$$
\text{spatial data} \;=\; \underbrace{\text{geometry}}_{\text{points, lines, polygons}} \;+\; \underbrace{\text{attributes}}_{\text{a table: one row per feature}} \;+\; \underbrace{\text{CRS}}_{\text{what the numbers in the geometry mean}}.
$$

Beginners tend to look after the first two and forget the third; most of this chapter is about why that is a mistake.

### Support and the modifiable areal unit problem

A county value is not a measurement at a point; it is an *aggregate* over a region, typically
$$
Z_i=\frac{1}{|A_i|}\int_{A_i} Z(\mathbf{s})\,d\mathbf{s}\quad(\text{a mean})\qquad\text{or}\qquad Y_i=\int_{A_i}\lambda(\mathbf{s})\,d\mathbf{s}\quad(\text{an expected count}).
$$
Change the boundaries, or the level of aggregation, and the numbers — and any correlation computed from them — change.
This is the **modifiable areal unit problem (MAUP)** (Openshaw, 1984). Keep it in mind whenever a result depends on a particular partition of space.


## 5. Spatial Data Containers (`spatialstats.core`)

### Loading data with GeoPandas

PySpatialStats does not invent a new data format. Vector data are read with GeoPandas into a `GeoDataFrame`:
an ordinary pandas table with one special column, `geometry`. We use **diabetes prevalence for 217 counties in the northeastern
United States** as the running example of a lattice.


In [ ]:
ne = gpd.read_file(DATA / "diabetes_northeast.shp")
print(f"{type(ne).__name__}: {ne.shape[0]} rows x {ne.shape[1]} columns | CRS: {ne.crs.name} (EPSG:{ne.crs.to_epsg()})")
print("geometry types:", ne.geom_type.value_counts().to_dict())
fields = pd.DataFrame({
    "dtype": ne.drop(columns="geometry").dtypes.astype(str),
    "example": ne.drop(columns="geometry").iloc[0].astype(str),
})
fields["meaning"] = pd.Series({
    "FIPS": "5-digit county code (numeric)", "STATE": "state name", "COUNTY": "county name",
    "x": "county centroid, easting (m)", "y": "county centroid, northing (m)", "POP_Total": "population",
    "Dia_count": "number of diabetes cases", "Dia_pct": "diabetes prevalence (%)", "Obesity": "obesity prevalence (%)",
    "Acc_Exer": "access to exercise opportunities (%)", "PhysInact": "physical inactivity (%)",
    "PCP_rate": "primary-care physicians per 100k", "FoodEnvIdx": "food environment index (0-10)",
    "SVI": "social vulnerability index", "UrbRural": "'Urban' / 'Rural'"})
fields

## 6. `SpatialData`: a Table with Roles

Statistical code needs to know *which column is the response, which are covariates, and where the coordinates are*.
`SpatialData` wraps a `GeoDataFrame` and records these **roles**, so downstream code does not have to guess and you
do not have to keep slicing columns by hand. It is deliberately thin: `sd.gdf` is always your GeoDataFrame.


In [ ]:
from spatialstats.core import SpatialData

sd = SpatialData(ne, y="Dia_pct", X=["Obesity", "PhysInact", "SVI"])
print(sd)                                        # note: geometry type is reported as 'Mixed' (why?)
print()
print("y_array()      ->", sd.y_array().shape, "  first values:", sd.y_array()[:4])
print("X_array()      ->", sd.X_array().shape)
print("coords_array() ->", sd.coords_array().shape, " first row:", sd.coords_array()[0].round(1))
print("crs            ->", sd.crs.name)

Two observations, both worth internalising:

* `geometry_type` says **`Mixed`** because the layer holds both `Polygon` and `MultiPolygon` features (counties with islands or detached parts). It is not an error — but code that assumes one geometry type will break on such data.
* `coords_array()` needs one $(x,y)$ location per county. For polygons it returns a **representative point** — a point guaranteed to lie *inside* the polygon — **not the centroid**. The docstring of the method speaks of centroids; the function actually called is `coords_from_geometry`, which uses `representative_point()`. The difference matters, as the next cell shows.

### Centroid versus representative point

The centroid is the polygon's centre of mass, $\bar{\mathbf s}=\frac{1}{|A|}\int_A \mathbf{s}\,d\mathbf{s}$. For a concave shape (a crescent, a coastline, a county wrapped around a bay) it can lie *outside* the polygon. A representative point is always inside, but is not the centre of mass.
The shapefile carries true centroids in the `x`, `y` columns, so we can measure the gap.

In [ ]:
cent = ne[["x", "y"]].to_numpy()                     # true centroids stored in the file
rep = sd.coords_array()                              # representative points used by SpatialData
gap_km = np.linalg.norm(rep - cent, axis=1) / 1000
outside = ~gpd.GeoSeries(gpd.points_from_xy(cent[:, 0], cent[:, 1]), crs=ne.crs).within(ne.geometry).to_numpy()

display(Markdown(
    f"**Distance between centroid and representative point** over {len(ne)} counties: "
    f"median {np.median(gap_km):.1f} km, mean {gap_km.mean():.1f} km, maximum {gap_km.max():.1f} km. "
    f"**{outside.sum()} centroids fall outside their own county.**"))

worst = int(gap_km.argmax())
fig, ax = plt.subplots(1, 2, figsize=(11, 4.4))
ne.plot(ax=ax[0], color="#edf2f7", edgecolor="#a0aec0", linewidth=0.3)
ne.iloc[[worst]].plot(ax=ax[0], color="#bee3f8", edgecolor="k")
x0_, y0_, x1_, y1_ = ne.iloc[worst].geometry.bounds                   # zoom to the county, keep its neighbours as context
pad = 0.12 * max(x1_ - x0_, y1_ - y0_)
ax[0].set_xlim(x0_ - pad, x1_ + pad); ax[0].set_ylim(y0_ - pad, y1_ + pad)
ax[0].scatter(*cent[worst], c=C["red"], s=50, zorder=5, label="centroid")
ax[0].scatter(*rep[worst], c=C["green"], s=50, marker="s", zorder=5, label="representative point")
ax[0].set_title(f"Largest gap: {ne.COUNTY.iloc[worst]}, {ne.STATE.iloc[worst]}")
ax[0].legend(loc="lower left"); ax[0].set_axis_off()
ax[1].hist(gap_km, bins=30, color=C["blue"], alpha=0.85)
ax[1].set_xlabel("centroid – representative point distance (km)"); ax[1].set_ylabel("counties")
ax[1].set_title("The two coordinates are not interchangeable")
plt.tight_layout(); plt.show()

**Practical rule:** if a method is sensitive to the *precise* location (distance-band weights, kernels, kriging), decide which point you mean, and tell `SpatialData` explicitly with the `coords` role.

In [ ]:
# Option 1: tell SpatialData to use the centroid *columns* stored in the file (a separate object, so `sd` stays untouched)
sd_cols = SpatialData(ne, y="Dia_pct").set_roles(coords=["x", "y"])
print("coords role = ['x', 'y'] -> equals the stored centroids:", np.allclose(sd_cols.coords_array(), cent))

# Option 2: build SpatialData from a plain table with coordinate columns; the points get their geometry from x, y
table = ne.drop(columns="geometry")
sd_pts = SpatialData.from_frame(table, x_col="x", y_col="y", crs=ne.crs, y="Dia_pct")
print(sd_pts)
print("from_frame also sets the coords role:", sd_pts.roles["coords"])

## 7. Reprojection Helpers

`SpatialData.to_crs()` returns a *new* object (the original is untouched) with roles preserved, and `ensure_projected()`
converts geographic (latitude/longitude) data to a projected CRS but leaves already-projected data alone. We meet the reasons
for both in section 0.5.

> **A trap worth knowing.** Reprojection transforms the **geometry**. If the `coords` role points at *columns* — because you set it, or because
> `from_frame()` did — those columns are copied unchanged, and `coords_array()` keeps returning the **old, un-reprojected numbers**. The last three lines of the next cell show it.
> When you reproject a `SpatialData`, either rely on the geometry (leave the `coords` role unset) or rebuild the coordinate columns.


In [ ]:
sd_ll = sd.to_crs(4326)                                # degrees
print("to_crs(4326)        :", sd_ll.crs.name, "| first coordinates:", sd_ll.coords_array()[0].round(4), "(lon, lat)")
print("roles preserved     :", {k: v for k, v in sd_ll.roles.items() if v})
print("ensure_projected() on already-projected data ->", sd.ensure_projected().crs.name, "(unchanged)")
print("ensure_projected() on lon/lat data          ->", sd_ll.ensure_projected().crs.name, "(default target!)")

# --- The trap: a coords role that names COLUMNS is not reprojected ------------------------------
for label, obj in [("coords role = ['x','y'] columns", sd_cols), ("built with from_frame()", sd_pts)]:
    moved = obj.to_crs(4326)
    print(f"\n{label}: geometry is now {moved.crs.name}, but coords_array()[0] = {moved.coords_array()[0].round(1)}")
fixed = sd_cols.to_crs(4326)
fixed.roles["coords"] = None                             # drop the stale role -> coordinates come from the geometry again
print("\nafter dropping the stale role         : coords_array()[0] =", fixed.coords_array()[0].round(4), "(lon, lat)")

***
## 8. Summary and Where Next

* **Data types.** Fields $Z(\mathbf{s})$, lattices $\{Z_i\}$ with weights $\mathbf{W}$, and point
  patterns with intensity $\lambda(\mathbf{s})$ call for different models; the areal unit is part of the
  data (MAUP).
* **`SpatialData`.** Roles make the table self-describing; `coords_array()` returns representative
  points unless you set the `coords` role explicitly — do not assume centroids.

**Previous:** [0.1 Installation and the Package Map](ch00_1_installation_and_package_map.ipynb)

**Next:** [0.3 Coordinate Reference Systems and Distances](ch00_3_crs_and_distances.ipynb) —
projection distortion, the Buffalo CRS case study, and distance metrics.

Full single-file chapter: [`ch00_setup_and_foundations.ipynb`](ch00_setup_and_foundations.ipynb).
